# Donut Filter: Binary CNN for Out-of-Focus Particle Detection

This notebook trains a lightweight binary CNN to distinguish **donut particles**
(out-of-focus ring artifacts) from all other particle types (liquid, solid, noise).

The model is designed to run *before* the phase classifier in the inference pipeline.
Particles predicted as donuts are excluded from phase classification entirely.

**Why image-only?**  
Donut morphology (a bright ring with a hollow center) is visually distinctive — the CNN
learns this pattern directly from the 128×128 image. No scalar features are needed.

**Data:**
- Positive (donut): 2,230 images from `cgwaves_particle_images_filtered/donut/`
- Negative (not-donut): liquid + solid + noise images from the same dataset

**Output:** `donut_filter_results/donut_filter_model.keras`

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import random
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    SpatialDropout2D, Flatten, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(f"TensorFlow {tf.__version__}")
print(f"NumPy      {np.__version__}")

## 2. Configuration

In [ ]:
# ── Google Colab setup ────────────────────────────────────────────────────────
# On Colab: run this cell. Locally: base_path stays './'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/GEOG5100/aircraft_ml/'
    IN_COLAB = True
except ImportError:
    base_path = './'
    IN_COLAB = False

print(f"{'Colab' if IN_COLAB else 'Local'} mode  |  base_path = {base_path}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGE_BASE   = Path(base_path) / 'particle_images_filtered'
SAVE_DIR     = Path(base_path) / 'donut_filter_results'


# particle_df.csv from the same dataset (has phase labels and particle_idx_seq)
PARTICLE_CSV = Path(base_path) / 'particle_df.csv'

# Where to save the trained model
SAVE_DIR.mkdir(exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
RANDOM_SEED  = 42
IMAGE_SIZE   = (128, 128)
BATCH_SIZE   = 32
EPOCHS       = 40
LEARNING_RATE = 1e-4
L2_REG       = 1e-4

# ── Reproducibility ───────────────────────────────────────────────────────────
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Saving model to: {SAVE_DIR}")

## 3. Load Data

Build file-path lists and binary labels from `particle_df.csv`.

| Phase | Label in CSV | Binary label |
|-------|-------------|---------------|
| 0 — Liquid | 0 | 0 (not donut) |
| 1 — Solid  | 1 | 0 (not donut) |
| 2 — Donut  | 2 | **1 (donut)** |
| 3 — Noise  | 3 | 0 (not donut) |

In [ ]:
PHASE_DIRS = {0: 'liquid', 1: 'solid', 2: 'donut', 3: 'noise'}

df = pd.read_csv(PARTICLE_CSV)
print(f"Total particles in CSV: {len(df):,}")
print(f"Phase distribution:")
print(df['phase'].value_counts().sort_index())

# Build image paths
def make_path(row):
    subdir = PHASE_DIRS[row['phase']]
    return str(IMAGE_BASE / subdir / f"particle_{row['particle_idx_seq']}.png")

df['image_path'] = df.apply(make_path, axis=1)

# Filter to images that actually exist on disk
exists = df['image_path'].apply(os.path.exists)
df = df[exists].reset_index(drop=True)
print(f"\nImages found on disk: {len(df):,}  (dropped {(~exists).sum()} missing)")

# Binary label: 1 = donut, 0 = not donut
df['binary_label'] = (df['phase'] == 2).astype(int)
print(f"\nBinary distribution:")
print(df['binary_label'].value_counts())
print(f"  0 — not donut : {(df['binary_label']==0).sum():,}")
print(f"  1 — donut     : {(df['binary_label']==1).sum():,}")

## 4. Train / Val / Test Split

In [ ]:
image_paths = df['image_path'].values
labels      = df['binary_label'].values

# 80 / 10 / 10 stratified split
paths_temp, paths_test, y_temp, y_test = train_test_split(
    image_paths, labels, test_size=0.10, stratify=labels, random_state=RANDOM_SEED
)
paths_train, paths_val, y_train, y_val = train_test_split(
    paths_temp, y_temp, test_size=0.111, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"Train : {len(paths_train):,}  (donut={y_train.sum():,}, not={( y_train==0).sum():,})")
print(f"Val   : {len(paths_val):,}   (donut={y_val.sum():,},   not={(y_val==0).sum():,})")
print(f"Test  : {len(paths_test):,}   (donut={y_test.sum():,},   not={(y_test==0).sum():,})")

# Class weights to handle imbalance
weights = class_weight.compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1]), y=y_train
)
class_weight_dict = {0: weights[0], 1: weights[1]}
print(f"\nClass weights: {class_weight_dict}")

## 5. tf.data Pipeline

In [ ]:
AUGMENTATION_PROB = 0.5

def load_image(path, label, augment=False):
    """Load a 128×128 grayscale PNG and optionally augment it."""
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)   # → [0, 1]
    img = tf.image.resize(img, IMAGE_SIZE)

    if augment:
        if tf.random.uniform(()) < AUGMENTATION_PROB:
            img = tf.image.flip_left_right(img)
        if tf.random.uniform(()) < AUGMENTATION_PROB:
            img = tf.image.flip_up_down(img)
        # Random 90-degree rotation
        if tf.random.uniform(()) < AUGMENTATION_PROB:
            k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
            img = tf.image.rot90(img, k=k)

    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, tf.cast(label, tf.float32)


def make_dataset(paths, labels, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (paths.astype(str), labels.astype(np.int32))
    )
    ds = ds.map(
        lambda p, l: load_image(p, l, augment=augment),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(paths_train, y_train, augment=True,  shuffle=True)
val_ds   = make_dataset(paths_val,   y_val,   augment=False, shuffle=False)
test_ds  = make_dataset(paths_test,  y_test,  augment=False, shuffle=False)

print("Datasets created.")

# Sanity check: visualise one batch
for imgs, lbls in train_ds.take(1):
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for i, ax in enumerate(axes.flat):
        if i < len(imgs):
            ax.imshow(imgs[i, :, :, 0], cmap='gray')
            ax.set_title('Donut' if int(lbls[i]) == 1 else 'Not', fontsize=8)
            ax.axis('off')
    plt.suptitle('Sample training batch')
    plt.tight_layout()
    plt.show()

## 6. Model Architecture

Three-block CNN with a single sigmoid output unit.  
Simpler than the phase classifier — donut detection is a more distinctive visual task.

In [ ]:
def build_donut_filter_model(input_shape=(128, 128, 1)):
    """
    Lightweight binary CNN: predicts P(donut) in [0, 1].

    Three conv blocks (32 → 64 → 128 filters) followed by a
    dense head with sigmoid output.
    """
    inputs = Input(shape=input_shape, name='image_input')

    # Block 1
    x = Conv2D(32, (3, 3), activation='relu', padding='same',
               kernel_regularizer=regularizers.l2(L2_REG))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = SpatialDropout2D(0.2)(x)

    # Block 2
    x = Conv2D(64, (3, 3), activation='relu', padding='same',
               kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = SpatialDropout2D(0.2)(x)

    # Block 3
    x = Conv2D(128, (3, 3), activation='relu', padding='same',
               kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = SpatialDropout2D(0.25)(x)

    # Classification head
    x = Flatten()(x)
    x = Dense(64, activation='relu',
               kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid', name='donut_prob')(x)

    model = Model(inputs=inputs, outputs=outputs, name='DonutFilter')
    return model


model = build_donut_filter_model()
model.summary()

## 7. Train

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        str(SAVE_DIR / 'donut_filter_model.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, title in zip(
    axes,
    ['loss', 'accuracy', 'auc'],
    ['Loss', 'Accuracy', 'AUC']
):
    ax.plot(history.history[metric],     label='Train')
    ax.plot(history.history[f'val_{metric}'], label='Val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Evaluate on Test Set

In [ ]:
# Load best checkpoint
model = tf.keras.models.load_model(SAVE_DIR / 'donut_filter_model.keras')

# Predict
donut_probs = model.predict(test_ds, verbose=0).ravel()
y_pred      = (donut_probs >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Donut', 'Donut']))

cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not Donut', 'Donut'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Donut Filter — Test Set Confusion Matrix')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### Threshold Sensitivity

Higher threshold → fewer false positives (fewer real particles called donut).  
Lower threshold → fewer false negatives (fewer donuts slip through).  
The default threshold in the inference notebook is 0.5 — adjust `DONUT_THRESHOLD` there if needed.

In [ ]:
thresholds = np.linspace(0.1, 0.9, 17)
rows = []
for t in thresholds:
    pred_t = (donut_probs >= t).astype(int)
    tp = ((pred_t == 1) & (y_test == 1)).sum()
    fp = ((pred_t == 1) & (y_test == 0)).sum()
    fn = ((pred_t == 0) & (y_test == 1)).sum()
    tn = ((pred_t == 0) & (y_test == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    rows.append({'threshold': t, 'precision': precision,
                 'recall': recall, 'f1': f1,
                 'false_positives': fp, 'false_negatives': fn})

thresh_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision')
ax.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall')
ax.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1', linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='Default threshold (0.5)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Donut Filter: Precision / Recall / F1 vs. Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nThreshold sensitivity table:")
print(thresh_df.to_string(index=False, float_format='{:.3f}'.format))

## 10. Visual Inspection

Check the highest-confidence false positives (real particles the model calls donut)
and false negatives (donuts that slipped through).

In [ ]:
def show_errors(paths, y_true, y_pred_prob, threshold=0.5, error_type='fp', n=8, title=''):
    """error_type: 'fp' = false positives, 'fn' = false negatives."""
    y_pred_bin = (y_pred_prob >= threshold).astype(int)
    if error_type == 'fp':
        mask = (y_pred_bin == 1) & (y_true == 0)   # predicted donut, actually not
        sort_key = y_pred_prob
    else:
        mask = (y_pred_bin == 0) & (y_true == 1)   # predicted not donut, actually donut
        sort_key = 1.0 - y_pred_prob

    idx = np.where(mask)[0]
    if len(idx) == 0:
        print(f"No {error_type} errors found.")
        return

    # Sort by worst first
    idx = idx[np.argsort(sort_key[idx])[::-1][:n]]

    n_show = min(n, len(idx))
    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for ax, i in zip(axes, idx):
        img = tf.image.decode_png(tf.io.read_file(paths[i]), channels=1).numpy()[:, :, 0]
        ax.imshow(img, cmap='gray')
        ax.set_title(f'p={y_pred_prob[i]:.2f}', fontsize=8)
        ax.axis('off')
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


show_errors(paths_test, y_test, donut_probs, error_type='fp',
            title='False Positives: real particles called donut (worst first)')

show_errors(paths_test, y_test, donut_probs, error_type='fn',
            title='False Negatives: donuts that slipped through (worst first)')

## 11. Save Outputs

The model is already saved at `donut_filter_results/donut_filter_model.keras`  
by the `ModelCheckpoint` callback. The cell below summarises what was saved.

In [ ]:
for f in sorted(SAVE_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40}  {size_kb:>8.1f} KB")

print(f"\nTo use this model in the inference notebook:")
print(f"  Set  DONUT_FILTER = True")
print(f"  Set  DONUT_MODEL_PATH = '{SAVE_DIR / 'donut_filter_model.keras'}'")
print(f"  Adjust DONUT_THRESHOLD (default 0.5) using the sensitivity plot above.")